In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 문서 로드

In [3]:
file_path = "../dataset/서울시 청년월세지원정책.pdf"

print("문서 로드중")
loader = PyPDFLoader(file_path)
docs = loader.load()

문서 로드중


In [4]:
len(docs)

10

In [5]:
docs[1].page_content

'- 2 -\n신청접수□  ㅇ 신청기간 : 수 화 마감2025. 6. 11.( ) 10:00 ~ 6. 24.( ) 18:00 ( ) ㅇ 선정인원 명: 15,000 ㅇ 신청방법 서울주거포털 : 온라인 신청선정 및 지급□  ㅇ 선정방법 임차보증금 월세 및 소득기준 개 구간으로 나누어 선정하고 : · 4인원 초과시 구간별 전산 추첨  ㅇ 지급방법 격월 일 전후로 계좌 입금: 25   -지급일정 월 : 10월 월 월(7 , 8 , 9 )월, 12월 월분(10 , 11 ) 지급 이후, 잔여 개월분은 7’년에 격월 지급 예정26  ( 지급일정 변동 가능※ )     자세한 지급 일정은 매 회차 지급 전 서울주거포털 공지사항 참고※    -지원대상자 최종 선정 후 자격 유지 여부 확인 심사( )후 지급 신청 자격요건2. ․자격 요건□ ㅇ 가구 주민등록상 신청인 인만 등록되어 있는 무주택자 청년 인 가구 ( ) 1 1주민등록등본상 청년이 아닌 세대원이 있는 경우 사업 신청대상에서 제외      ※       ※ 외국인은 신청 불가하며 재외국민의 경우 건강보험의 가입자 또는 피부양자이면 신청 가능,  ㅇ 주소 신청일 기준 서울에 주민등록이 되어 있고 실제 거주하는 청년( ) ,    -신청일 기준 임대차계약서상 임차건물 소재지에 주민등록 전입신고( )이 등재된 임차인본인이 신청해야 함부모 형제 친구 등       , , ※ 타인의 명의로 임대차계약을 체결한 경우 신청 불가      ※ 주민등록등본상 청년인 형제자매 및 동거인이 있는 경우 임차인 명의의 인에 한해 신청 가능‘ ’ , 1공유주택 쉐어하우스 등에 거주하며 임대인 사업자 포함 과 개별 임대차 계약을 체결한       ( ) ( )※ 주민등록등본상 동거인은 동시 지원 신청 가능'

# 2. 텍스트 스플리터(청킹 = 조각내다)

In [6]:
# 1. RecursiveCharacterTextSplitter 객체 생성
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,     # 한 조각(chunk)의 최대 문자 수 (대략 1000자 단위로 자름)
    chunk_overlap = 100,   # 조각 간에 겹치는 문자 수 (문맥 유지를 위해 앞뒤로 100자 겹치게 함)
    separators = ["\n\n", "\n", " ", ""]  # 텍스트를 자를 때 우선적으로 고려할 구분자들 (단락 → 줄바꿈 → 공백 → 글자 단위)
)

# 2. 문서(docs)를 여러 개의 chunk로 분리
chunks = splitter.split_documents(docs)

# 3. 전체 잘린 조각(chunk)의 개수 출력
print(len(chunks), "전체 잘린 chunk 사이즈")

14 전체 잘린 chunk 사이즈


In [7]:
print(chunks[1].page_content)

- 2 -
신청접수□  ㅇ 신청기간 : 수 화 마감2025. 6. 11.( ) 10:00 ~ 6. 24.( ) 18:00 ( ) ㅇ 선정인원 명: 15,000 ㅇ 신청방법 서울주거포털 : 온라인 신청선정 및 지급□  ㅇ 선정방법 임차보증금 월세 및 소득기준 개 구간으로 나누어 선정하고 : · 4인원 초과시 구간별 전산 추첨  ㅇ 지급방법 격월 일 전후로 계좌 입금: 25   -지급일정 월 : 10월 월 월(7 , 8 , 9 )월, 12월 월분(10 , 11 ) 지급 이후, 잔여 개월분은 7’년에 격월 지급 예정26  ( 지급일정 변동 가능※ )     자세한 지급 일정은 매 회차 지급 전 서울주거포털 공지사항 참고※    -지원대상자 최종 선정 후 자격 유지 여부 확인 심사( )후 지급 신청 자격요건2. ․자격 요건□ ㅇ 가구 주민등록상 신청인 인만 등록되어 있는 무주택자 청년 인 가구 ( ) 1 1주민등록등본상 청년이 아닌 세대원이 있는 경우 사업 신청대상에서 제외      ※       ※ 외국인은 신청 불가하며 재외국민의 경우 건강보험의 가입자 또는 피부양자이면 신청 가능,  ㅇ 주소 신청일 기준 서울에 주민등록이 되어 있고 실제 거주하는 청년( ) ,    -신청일 기준 임대차계약서상 임차건물 소재지에 주민등록 전입신고( )이 등재된 임차인본인이 신청해야 함부모 형제 친구 등       , , ※ 타인의 명의로 임대차계약을 체결한 경우 신청 불가      ※ 주민등록등본상 청년인 형제자매 및 동거인이 있는 경우 임차인 명의의 인에 한해 신청 가능‘ ’ , 1공유주택 쉐어하우스 등에 거주하며 임대인 사업자 포함 과 개별 임대차 계약을 체결한       ( ) ( )※ 주민등록등본상 동거인은 동시 지원 신청 가능


# 3. 임베딩 생성 chromadb 저장

In [8]:
chunks[1].metadata

{'producer': 'Microsoft: Print To PDF',
 'creator': 'PyPDF',
 'creationdate': '2025-06-02T15:06:47+09:00',
 'author': '',
 'moddate': '2025-06-02T15:06:47+09:00',
 'title': '〲㔲⁄렜⃜䒭㣔타ꠠ⃑\ue0f5⸸睨硰',
 'source': '../dataset/서울시 청년월세지원정책.pdf',
 'total_pages': 10,
 'page': 1,
 'page_label': '2'}

In [9]:
db_path = "../vectorstore/chromadb_project"   # 벡터 저장소를 저장할 경로 지정

embedding = OpenAIEmbeddings(model="text-embedding-3-small")   # 텍스트를 벡터로 변환할 임베딩 모델 설정

collection_name = "monthly_youth_support_policy"

In [10]:
vectorstore = Chroma.from_documents(   # 문서 조각(chunks)을 벡터화하여 저장소 생성
    documents=chunks,                  # 분할된 문서 조각 리스트
    embedding=embedding,               # 사용할 임베딩 모델
    persist_directory=db_path,         # 벡터 데이터를 저장할 폴더 경로
    collection_name=collection_name     # 데이터 컬렉션 이름 지정
)

print("벡터 저장소 저장 완료")   # 벡터 저장소 생성이 완료되었음을 출력

벡터 저장소 저장 완료


# 4. 검색기 구성(retriever)

In [ ]:
retriever = vectorstore.as_retriever(        # 벡터 저장소를 검색기(retriever)로 변환
    search_type = "mmr",
    search_kwargs = {"k" : 5,
                     "fetch_k" : 20,
                     "lambda_mult" : 0.6
                     }
) 
result = retriever.invoke("부동산 중개보수 및 이사비 지원에 대한 내용을 요약해줘.")   # 질문과 가장 관련 있는 문서 조각 검색
result   # 검색된 문서 조각(결과) 출력

In [ ]:
for item in result:    # 검색 결과(result)에 있는 각 문서 조각을 하나씩 반복
    print(item.page_content[:50])  # 각 조각의 텍스트 앞 50글자만 출력
    print("-"*50)      # 구분선을 출력해 각 조각을 시각적으로 구분

# 5. 기본 체인 만들기

In [ ]:
# 1. 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system","""당신은 Suggest Agent 입니다.
    역할은 사용자의 개인·가구 정보(예: 연령, 주소(시/구), 소득(월/연), 가구원 수, 거주형태(전세/월세), 보증금/월세 금액, 근로상태 등)
    를 입력받아, 내부 RAG 파이프라인(ChromaDB에 저장된 공적 문서: 서울청년포털, 복지로, 서울시 고시공고 등)
    을 검색하여 지원 가능성, 지원 금액, 필요 조건(필수 제출 서류 / 선택 제출 서류 구분), 지원 기간/횟수, '적용 제외 조건,
    관련 신청 링크/문의처'를 구조화된 JSON으로 반환하는 것입니다.

    작업 규칙:
    1. 입력 언어: 한국어. 출력도 한국어.
    2. RAG 검색 결과(검색에 사용된 문서 리스트)는 반드시 포함하고, 각 항목에 대해 출처 URL을 제공하세요.
    3. 결과는 기계가 처리할 수 있는 JSON 구조(아래 예시 포맷)를 엄격히 따르세요. 추가 설명은 JSON 바깥에 간단히 1-2문장만 허용합니다.
    4. 정책 근거(문서에서 직접 인용한 문장)는 `raw_snippet` 필드에 1~2문장으로 제공하고, 인용 출처(URL)와 발행일을 함께 제공하세요. 인용은 25단어 이내로 요약(직접 인용 형태가 필요하면 따옴표로 표기).
    5. 불확실한 항목은 `confidence`(0.0~1.0)로 표기하고, 불확실 이유를 `notes`에 간단히 기재하세요.
    6. 우대(선택) 조건과 필수 제출 서류는 **분리**해서 반환하세요.
    7. 개인정보는 포함하지 마세요(예: 주민등록번호 등). 사용자의 입력은 Agent 내부에서만 사용되고 외부로 노출되지 않는다고 가정합니다.
    8. 답변 시 이모지 사용 금지.
    9. 응답은 최신 문서(문서의 `last_updated` 또는 발행일 기준으로) 우선으로 판단하되, 불확실하면 출처의 발행일을 명시하세요.
    10. 출력 JSON의 `policy_recommendations` 항목에는 각 정책 건마다 `source_urls`(우선순위 높은 3개 이내)와 `most_relevant_snippet`을 포함해야 합니다.

    추가 지시:
    - 검색엔진(retriever)은 MMR 추천. 후보(fetch_k)는 충분히 가져온 뒤(k=최종) 다양성과 관련성 균형(lambda_mult)을 고려해 선택하세요.
    - 반환할 우선순위는 '지원 가능성 높은 것 → 금액·기간이 큰 것 → 신청 난이도(서류 적음) 순'으로 정렬하세요."


     [context]
     {context}
     """),
     ("human", "{question}")
])

# 모델 선택
model = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature= 0
)

# 3. outputparser 선택
outputparser = StrOutputParser()

# 4. chain 생성
chain = rag_prompt | model | outputparser

# 6. RAG 체인 만들기

In [ ]:
# 문서 조각들을 하나의 문자열로 합치는 함수
def format_docs(docs):
    tmp_docs = []                           # 각 문서 조각의 텍스트를 임시로 저장할 리스트
    for item in docs:                       # docs 리스트의 각 문서 조각 반복
        tmp_docs.append(item.page_content)  # 각 조각의 텍스트(page_content)만 추출해 리스트에 추가
    return "\n\n---\n\n".join(tmp_docs)     # 문서 조각들을 구분선("\n\n---\n\n")으로 연결하여 하나의 문자열로 반환

In [ ]:
# rag_chain 만들기
rag_chain = (
    {"context" : RunnableLambda(lambda x : x["question"]) | retriever | RunnableLambda(format_docs),
     "question" : RunnableLambda(lambda x : x["question"])
    }
    | chain
)
rag_chain

In [ ]:
result = rag_chain.invoke(
    {
     "question" :"28세 서울에 거주하고 있는 남자이고 결혼은 하지 않았고 이번에 이사가려고 하는데 이사비 지원을 받을 수 있을까요?"
    }
)
print(result)